In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [3]:
train_dataset = datasets.CIFAR10(
    root="./data/cifar10",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data/cifar10",
    train=False,
    download=False,
    transform=transform
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [5]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([16, 3, 224, 224])
torch.Size([16])


In [6]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [8]:
out = model(images.to(device))
print(out.shape)

torch.Size([16, 10])


In [9]:
images, labels = next(iter(train_loader))
out = model(images.to(device))
print(out.shape)

torch.Size([16, 10])


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [11]:
print("training started")

for epoch in range(3):
    print("epoch start:", epoch)

    model.train()
    running_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 50 == 0:
            print("batch", i, "loss", loss.item())

    print(f"Epoch {epoch+1} done, loss: {running_loss:.4f}")

training started
epoch start: 0
batch 0 loss 2.555826187133789
batch 50 loss 1.350637674331665
batch 100 loss 1.3846282958984375
batch 150 loss 1.6538892984390259
batch 200 loss 1.4469062089920044
batch 250 loss 1.119105339050293
batch 300 loss 1.3284577131271362
batch 350 loss 0.9974024295806885
batch 400 loss 1.278262972831726
batch 450 loss 0.9338995814323425
batch 500 loss 1.0249783992767334
batch 550 loss 1.2001972198486328
batch 600 loss 1.1790622472763062
batch 650 loss 1.3016669750213623
batch 700 loss 1.2165977954864502
batch 750 loss 1.255006194114685
batch 800 loss 0.8187512755393982
batch 850 loss 0.5318285822868347
batch 900 loss 1.700400948524475
batch 950 loss 1.5815874338150024
batch 1000 loss 1.082075595855713
batch 1050 loss 0.7930273413658142
batch 1100 loss 1.5528156757354736
batch 1150 loss 0.6545356512069702
batch 1200 loss 0.43118008971214294
batch 1250 loss 0.6947823166847229
batch 1300 loss 0.4491015672683716
batch 1350 loss 0.8311031460762024
batch 1400 loss 0

validation


In [13]:
from torch.utils.data import random_split

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

In [14]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("val accuracy:", correct / total)

val accuracy: 0.9338
